# 18j Gamma metadata settlement-family audit

Run this notebook from inside the repository, preferably after copying it to `notebooks/18j_gamma_metadata_settlement_family_audit.ipynb`.

It classifies Hong Kong Polymarket temperature contracts into HKO Daily Extract one-decimal, Wunderground / airport, ambiguous or other settlement families. It also creates `data/review_bundles/18j_review_bundle.zip`, which you can download from Jupyter and upload back for inspection.

In [ ]:
from pathlib import Path
import os, sys, subprocess, pandas as pd

cwd = Path.cwd()
repo = cwd.parent if cwd.name == 'notebooks' else cwd
os.chdir(repo)
print('Repository root:', Path.cwd())
print('Script exists:', Path('scripts/18j_gamma_metadata_settlement_family_audit.py').exists())

## Run the audit

This may take a few minutes because it calls the Gamma API and caches raw JSON responses under `data/raw/polymarket_gamma_18j/`.

In [ ]:
result = subprocess.run(
    [sys.executable, 'scripts/18j_gamma_metadata_settlement_family_audit.py'],
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print('STDERR:')
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'18j script failed with return code {result.returncode}')

## Inspect core outputs

In [ ]:
outputs = [
    'data/processed/18j_gamma_metadata_settlement_family_audit.csv',
    'data/processed/18j_full_hko_contract_event_universe_preliminary.csv',
    'data/processed/18j_excluded_or_ambiguous_hk_contracts.csv',
    'data/processed/18j_settlement_family_summary.csv',
    'data/processed/18j_contract_event_type_summary.csv',
    'data/processed/18j_march13_settlement_family_audit.csv',
    'docs/research_outputs/18j_gamma_metadata_settlement_family_audit_report.md',
    'data/review_bundles/18j_review_bundle.zip',
]

for f in outputs:
    p = Path(f)
    if not p.exists():
        print(f'{f}: MISSING')
    elif p.suffix == '.csv':
        df = pd.read_csv(p)
        print(f'{f}: {df.shape}')
    else:
        print(f'{f}: FOUND ({p.stat().st_size} bytes)')

print('\nSettlement-family summary')
display(pd.read_csv('data/processed/18j_settlement_family_summary.csv'))

print('\nContract-event-type summary')
display(pd.read_csv('data/processed/18j_contract_event_type_summary.csv'))

## Inspect March 13 and admissible HKO contracts

In [ ]:
march13 = pd.read_csv('data/processed/18j_march13_settlement_family_audit.csv')
print('March 13 rows:', march13.shape)
if not march13.empty:
    cols = [c for c in [
        'event_date', 'market_slug', 'market_question', 'group_item_title',
        'settlement_family', 'contract_event_type', 'event_set',
        'admissible_hko_event_contract', 'settlement_family_reason'
    ] if c in march13.columns]
    display(march13[cols])

universe = pd.read_csv('data/processed/18j_full_hko_contract_event_universe_preliminary.csv')
print('Preliminary admissible HKO event contracts:', universe.shape)
if not universe.empty:
    cols = [c for c in [
        'event_date', 'market_slug', 'market_question', 'group_item_title',
        'contract_event_type', 'event_set', 'settlement_family', 'selected_yes_token_id'
    ] if c in universe.columns]
    display(universe[cols].head(100))

## Report preview and download bundle

In [ ]:
from IPython.display import FileLink, display, Markdown

report = Path('docs/research_outputs/18j_gamma_metadata_settlement_family_audit_report.md')
if report.exists():
    text = report.read_text(encoding='utf-8')
    display(Markdown(text[:8000]))

bundle = Path('data/review_bundles/18j_review_bundle.zip')
if bundle.exists():
    print('Download this zip from Jupyter, then upload it back to ChatGPT for inspection:')
    display(FileLink(str(bundle)))
else:
    print('Bundle missing.')